# Gravity Model for Bird Transition Data

This notebook demonstrates how to use the gravity model script to analyze bird movement patterns between RFID feeders.

## Overview

The gravity model analyzes Origin-Destination (OD) flows between feeders, modeling transition counts as a function of:
- Distance between feeders (distance-decay)
- Origin and destination attractiveness (total visits)
- Time of day (morning/afternoon/evening/night)
- Species identity
- Seasonal effects (day of year)

The model is implemented as a Generalized Linear Model (GLM) with Poisson or Negative-Binomial distribution.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Import functions from gravity_model module
from gravity_model import (
    generate_sample_data,
    load_data,
    build_transitions,
    compute_distance_matrix,
    aggregate_od,
    fit_poisson_model,
    fit_negative_binomial_model,
    select_best_model,
    compute_residuals,
    cross_validate_model,
    plot_diagnostics,
    plot_distance_decay,
    plot_flow_map,
    save_model_outputs,
    print_model_summary
)

# Configure plotting
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## 1. Load or Generate Data

You can either:
- Load real data from CSV files (bouts.csv and feeders.csv)
- Generate sample data for testing

### Option A: Generate Sample Data

In [ ]:
# Generate sample data
bouts_df, feeders_df = generate_sample_data(
    n_birds=50,
    n_feeders=10,
    n_bouts=1000,
    seed=42
)

print(f"Generated {len(bouts_df)} bouts for {bouts_df['bird_id'].nunique()} birds")
print(f"Species: {bouts_df['species'].unique()}")
print(f"\nFeeder locations: {len(feeders_df)} feeders")

# Display sample data
display(bouts_df.head())
display(feeders_df.head())

### Option B: Load Real Data (uncomment to use)

```python
# Load from CSV files
bouts_df, feeders_df = load_data(
    bouts_path='path/to/bouts.csv',
    feeders_path='path/to/feeders.csv'
)
```

## 2. Build Transitions

Create consecutive transitions from bout data for each bird.

In [ ]:
# Build transitions (consecutive bouts for each bird)
transitions_df = build_transitions(bouts_df, exclude_self_loops=True)

print(f"Created {len(transitions_df)} transitions")
print(f"\nTransitions by species:")
print(transitions_df['species'].value_counts())

# Display sample transitions
display(transitions_df.head(10))

## 3. Compute Distances

Calculate Euclidean distances between all feeder pairs.

In [ ]:
# Compute distance matrix
distance_df = compute_distance_matrix(feeders_df)

print(f"Distance range: {distance_df['distance_m'].min():.1f} - {distance_df['distance_m'].max():.1f} metres")

# Plot distance distribution
plt.figure(figsize=(10, 6))
plt.hist(distance_df['distance_m'], bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Distance (metres)')
plt.ylabel('Frequency')
plt.title('Distribution of Inter-Feeder Distances')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Aggregate OD Flows

Aggregate transitions into Origin-Destination flows with exposure calculation.

In [ ]:
# Aggregate transitions into OD flow table
od_df = aggregate_od(
    transitions_df,
    bouts_df,
    distance_df,
    time_block='time_of_day'  # or None for no time stratification
)

print(f"Created {len(od_df)} OD flow records")
print(f"Total transitions: {od_df['n_transitions'].sum()}")
print(f"\nRecords with observed transitions: {(od_df['n_transitions'] > 0).sum()}")

# Display sample OD flows
display(od_df[od_df['n_transitions'] > 0].head(10))

## 5. Fit Gravity Models

Fit both Poisson and Negative-Binomial GLMs and select the best model.

In [ ]:
# Fit Poisson model
poisson_result = fit_poisson_model(od_df)

# Fit Negative Binomial model
nb_result = fit_negative_binomial_model(od_df)

# Select best model
best_model, model_name = select_best_model(poisson_result, nb_result)

print(f"\nSelected model: {model_name}")

## 6. Compute Residuals and Evaluate

In [ ]:
# Compute residuals
residuals_df = compute_residuals(od_df, best_model)

print("Residual statistics:")
print(residuals_df[['raw_residual', 'pearson_residual', 'deviance_residual']].describe())

# Display flows with largest residuals
print("\nTop 10 flows by absolute raw residual:")
display(residuals_df.nlargest(10, 'raw_residual')[[
    'origin', 'destination', 'species', 'observed', 'predicted', 'raw_residual'
]])

## 7. Cross-Validation

In [ ]:
# Perform cross-validation
formula = 'n_transitions ~ log_distance + log_origin_visits + log_dest_visits + day_of_year + C(time_of_day) + C(species)'

cv_results = cross_validate_model(
    od_df,
    formula=formula,
    n_folds=5,
    model_type=model_name
)

print(f"\nCross-validation results:")
print(f"  RMSE: {cv_results['rmse_mean']:.3f} ± {cv_results['rmse_std']:.3f}")
print(f"  MAE:  {cv_results['mae_mean']:.3f} ± {cv_results['mae_std']:.3f}")

## 8. Model Summary

In [ ]:
# Print comprehensive model summary
print_model_summary(best_model, model_name, cv_results)

## 9. Diagnostic Plots

In [ ]:
# Create output directory
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

# Plot diagnostics
plot_diagnostics(residuals_df, output_dir, model_name)

# Display the diagnostic plot
from IPython.display import Image
display(Image(filename=output_dir / f'{model_name}_diagnostics.png'))

## 10. Distance-Decay Plot

In [ ]:
# Plot distance-decay relationship
plot_distance_decay(residuals_df, best_model, output_dir, model_name)

# Display the plot
display(Image(filename=output_dir / f'{model_name}_distance_decay.png'))

## 11. Flow Map

In [ ]:
# Create flow map with residual edges
plot_flow_map(residuals_df, feeders_df, output_dir, model_name, top_n=20)

# Display the map
display(Image(filename=output_dir / f'{model_name}_flow_map.png'))

## 12. Save Model Outputs

In [ ]:
# Save coefficients and predictions to CSV
save_model_outputs(best_model, residuals_df, output_dir, model_name)

print(f"\nAll outputs saved to: {output_dir}")
print(f"\nFiles created:")
for file in sorted(output_dir.glob('*')):
    print(f"  - {file.name}")

## 13. Examine Model Coefficients

In [ ]:
# Load and display coefficients
coef_df = pd.read_csv(output_dir / f'{model_name}_coefficients.csv')

# Sort by absolute z-value to see most significant
coef_df['abs_z'] = coef_df['z_value'].abs()
coef_df_sorted = coef_df.sort_values('abs_z', ascending=False)

print("Model coefficients (sorted by significance):")
display(coef_df_sorted[['parameter', 'coefficient', 'std_error', 'z_value', 'p_value']])

## 14. Interpretation

### Key Model Parameters:

1. **Distance-decay (γ)**: The `log_distance` coefficient indicates how transitions decrease with distance. 
   - A negative coefficient means fewer transitions over longer distances (expected)
   - γ = -0.5 means ~40% reduction per log-unit distance

2. **Attractiveness**:
   - `log_origin_visits`: Origin feeder popularity effect
   - `log_dest_visits`: Destination feeder attractiveness

3. **Time of Day**: Coefficients show how transition rates vary by time period

4. **Species**: Fixed effects capture species-specific movement patterns

5. **Seasonal**: `day_of_year` captures temporal trends

### Model Quality:

- **AIC/BIC**: Lower values indicate better fit
- **Dispersion**: >1.5 suggests overdispersion (Negative-Binomial preferred)
- **Cross-validation RMSE**: Out-of-sample prediction error
- **Residual plots**: Should show no patterns (random scatter around zero)

## 15. Optional: Fit Per-Species Models

For more detailed species-specific analysis:

In [ ]:
# Fit separate models for each species
species_results = {}

for species in od_df['species'].unique():
    print(f"\n{'='*60}")
    print(f"SPECIES: {species}")
    print(f"{'='*60}")
    
    # Filter data for this species
    species_df = od_df[od_df['species'] == species].copy()
    
    # Fit models
    poisson_res = fit_poisson_model(species_df)
    nb_res = fit_negative_binomial_model(species_df)
    
    # Select best
    best, name = select_best_model(poisson_res, nb_res)
    
    if best is not None:
        species_results[species] = (best, name)
        
        # Print summary
        print_model_summary(best, f"{name} ({species})")
        
        # Save outputs
        species_dir = output_dir / species
        species_dir.mkdir(exist_ok=True)
        
        residuals = compute_residuals(species_df, best)
        save_model_outputs(best, residuals, species_dir, name)
        plot_diagnostics(residuals, species_dir, name)
        plot_distance_decay(residuals, best, species_dir, name)

print(f"\nPer-species models saved to {output_dir}/[species]/")

## Conclusion

This notebook demonstrated:

1. ✅ Loading/generating bird bout and feeder data
2. ✅ Building transitions from consecutive bouts
3. ✅ Aggregating OD flows with exposure calculation
4. ✅ Fitting Poisson and Negative-Binomial gravity models
5. ✅ Model selection via AIC/BIC and overdispersion tests
6. ✅ Cross-validation for out-of-sample performance
7. ✅ Residual analysis and diagnostic plots
8. ✅ Distance-decay visualization
9. ✅ Flow map with residual edges
10. ✅ Saving results to CSV and PNG files

### Next Steps:

- Apply to real RFID data from Wytham Woods
- Explore spatial autocorrelation in residuals
- Test alternative distance metrics (network distance vs Euclidean)
- Include environmental covariates (habitat, weather)
- Implement spatial smoothing for attractiveness terms